# Install Dependecies

In [7]:
%%capture
%pip install numpy
%pip install pandas
%pip install matplotlib.pyplot
%pip install python-terrier
%pip install "pyterrier-alpha[parallel]"
%pip install ipynbname
%pip install torch
%pip install transformers
%pip install accelerate
%pip install sentencepiece

In [2]:
# Load java
!curl -s "https://get.sdkman.io" | bash && source "$HOME/.sdkman/bin/sdkman-init.sh" && sdk install java 11.0.22-amzn < /dev/null

# check java and version
!which java
!java -version
!readlink -f $(which java)
!ls -la /usr/lib/jvm
!java --version
!javac --version


                                -+syyyyyyys:
                            `/yho:`       -yd.
                         `/yh/`             +m.
                       .oho.                 hy                          .`
                     .sh/`                   :N`                `-/o`  `+dyyo:.
                   .yh:`                     `M-          `-/osysoym  :hs` `-+sys:      hhyssssssssy+
                 .sh:`                       `N:          ms/-``  yy.yh-      -hy.    `.N-````````+N.
               `od/`                         `N-       -/oM-      ddd+`     `sd:     hNNm        -N:
              :do`                           .M.       dMMM-     `ms.      /d+`     `NMMs       `do
            .yy-                             :N`    ```mMMM.      -      -hy.       /MMM:       yh
          `+d+`           `:/oo/`       `-/osyh/ossssssdNMM`           .sh:         yMMN`      /m.
         -dh-           :ymNMMMMy  `-/shmNm-`:N/-.``   `.sN            /N-         `NMMy      .m/
  

# Imports

In [ ]:
import itertools
import json
import os
import re
import time
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import pyterrier as pt
from pathlib import Path
from tqdm.auto import tqdm
import pyterrier_alpha as pta
import ipynbname

import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from pyterrier.measures import nDCG, RR, P, R

In [9]:
ROOT_DIR = ipynbname.path().parent
ROOT_DIR = Path(ROOT_DIR)
print(ROOT_DIR)

/home/tlvj/msc_datalogi/2_semester/se/notebooks


# PyTerrier - Local

In [10]:
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk-amd64"
os.environ["JVM_PATH"] = "/usr/lib/jvm/java-11-openjdk-amd64/lib/server/libjvm.so"
os.environ["PATH"] = os.environ["JAVA_HOME"] + "/bin:" + os.environ["PATH"]

import pyterrier as pt

if not pt.java.started():
    pt.java.init()

print("JAVA_HOME:", os.environ["JAVA_HOME"])
print("JVM_PATH:", os.environ["JVM_PATH"])
print("Java started:", pt.java.started())

JAVA_HOME: /usr/lib/jvm/java-11-openjdk-amd64
JVM_PATH: /usr/lib/jvm/java-11-openjdk-amd64/lib/server/libjvm.so
Java started: True


# Load Dataset

In [11]:
base = Path.cwd() / "msc_datalogi" / "2_semester" / "se" / "project_handout"
docs          = pd.read_json(f'{base}/docs2.jsonl', lines=True, dtype={'docno': str})
train_queries = pd.read_csv(f'{base}/train_queries.csv')
train_qrels   = pd.read_csv(f'{base}/train_qrels.csv')

# Load Indexes

In [12]:
index_path_none      = str((ROOT_DIR / ".." / "indexes" / "full").resolve())
index_path_stop      = str((ROOT_DIR / ".." / "indexes" / "stopwords").resolve())
index_path_stem      = str((ROOT_DIR / ".." / "indexes" / "stemming").resolve())
index_path_stop_stem = str((ROOT_DIR / ".." / "indexes" / "stop-stem").resolve())

index_none      = pt.IndexFactory.of(index_path_none)
index_stop      = pt.IndexFactory.of(index_path_stop)
index_stem      = pt.IndexFactory.of(index_path_stem)
index_stop_stem = pt.IndexFactory.of(index_path_stop_stem)

indices = {
    "stopwords": index_stop,
    "stop_stem": index_stop_stem,
    "none": index_none,
    "stem": index_stem
}

# Preprocessing

In [14]:
if "text" in train_queries.columns and "query" not in train_queries.columns:
    train_queries = train_queries.rename(columns={"text": "query"})

train_queries["qid"]   = train_queries["qid"].astype(str)
train_queries["query"] = train_queries["query"].astype(str)
train_qrels["qid"]     = train_qrels["qid"].astype(str)
train_qrels["docno"]   = train_qrels["docno"].astype(str)

if "label" not in train_qrels.columns:
    train_qrels = train_qrels.rename(columns={"relevance": "label"} if "relevance" in train_qrels.columns else {"rel": "label"})

train_qrels["label"] = train_qrels["label"].astype(int)

# Cache

In [41]:
cache_dir = (ROOT_DIR / ".." / "results").resolve()
cache_dir.mkdir(parents=True, exist_ok=True)
bm25_cache_path = cache_dir / "bm25_tuning_results.json"
lm_cache_path   = cache_dir / "lm_tuning_results.json"

best_models = {name: {} for name in indices}

In [20]:
def sync_cache(cache_path, wmodel, ctrl_fn):
    if not cache_path.exists():
        return
    with open(cache_path) as f:
        cache = json.load(f)
    for index_name, best in cache["best_configs"].items():
        if index_name in indices:
            best_models[index_name][wmodel] = {
                "model": pt.terrier.Retriever(indices[index_name], wmodel=wmodel,
                                              controls=ctrl_fn(best["config"])),
                "config": best["config"], "score": best["score"]}

In [21]:
sync_cache(bm25_cache_path, "BM25",        lambda c: {"bm25.k_1": c["k1"], "bm25.b": c["b"]})
sync_cache(lm_cache_path,   "Hiemstra_LM", lambda c: {"c": c["c"]})

In [22]:
EVAL_MEASURE = "ndcg_cut_10"
best_summary_df = pd.DataFrame([
    {"index": i, "model": m, "best_config": info["config"], f"best_{EVAL_MEASURE}": round(info["score"], 4)}
    for i, md in best_models.items() for m, info in md.items()
]).sort_values(by=f"best_{EVAL_MEASURE}", ascending=False).reset_index(drop=True)
best_summary_df

,index,model,best_config,best_ndcg_cut_10
0,stop_stem,BM25,"{'k1': 0.9, 'b': 0.6}",0.4516
1,stopwords,BM25,"{'k1': 0.9, 'b': 0.6}",0.4434
2,stop_stem,Hiemstra_LM,{'c': 0.05},0.4411
3,stopwords,Hiemstra_LM,{'c': 0.05},0.4342


# Backbone + LM baseline retrievers

In [23]:
# Single best model-index = backbone for final retrieval of expanded queries.
prf_best = best_summary_df.iloc[0]
BACKBONE_INDEX_NAME = prf_best["index"]
BACKBONE_MODEL_NAME = prf_best["model"]
BACKBONE_CONFIG     = prf_best["best_config"]
backbone_index      = indices[BACKBONE_INDEX_NAME]

backbone_second = pt.terrier.Retriever(
    backbone_index, wmodel="BM25",
    controls={"bm25.k_1": BACKBONE_CONFIG["k1"], "bm25.b": BACKBONE_CONFIG["b"]},
)

# LM baseline (best Hiemstra_LM from Task 3) for the comparison table.
lm_best_row = best_summary_df[best_summary_df["model"] == "Hiemstra_LM"].iloc[0]
lm_baseline = pt.terrier.Retriever(
    indices[lm_best_row["index"]], wmodel="Hiemstra_LM",
    controls={"c": lm_best_row["best_config"]["c"]},
)
print(f"Backbone: {BACKBONE_MODEL_NAME}/{BACKBONE_INDEX_NAME} {BACKBONE_CONFIG}")

Backbone: BM25/stop_stem {'k1': 0.9, 'b': 0.6}


# 6 Query expansion with LLMs

We expand each query by prompting a free, instruction-tuned LLM (`google/flan-t5-base`)
in a zero-shot Query2doc style (Wang et al., 2023; the approach proposed in the lab):
generate a pseudo-document that answers the query, then append it to the query. Because the
generated passage is much longer than the query, we repeat the original query terms ~5× so
they keep their weight (the lab's `q' = Concat(q×5, d)` trick). The expanded queries are
retrieved with our single best backbone (BM25 / stop_stem), and compared against the BM25
and LM baselines on NDCG, MRR, Precision and Recall at 5/10/20.

# Load LLM

In [24]:
LLM_NAME = "google/flan-t5-base"          # task also suggests flan-t5-large / llmware/bling-1b-0.1
LLM_TAG  = LLM_NAME.split("/")[-1].replace("-", "_")
device   = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(LLM_NAME)
llm = AutoModelForSeq2SeqLM.from_pretrained(LLM_NAME).to(device)
llm.eval()
print(f"Loaded {LLM_NAME} on {device}")

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Loaded google/flan-t5-base on cuda


# generation + prompt builders

In [32]:
@torch.no_grad()
def llm_generate(prompt, max_new_tokens=128):   # CoT is longer -> more tokens
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to(device)
    out = llm.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)  # deterministic
    text = tokenizer.decode(out[0], skip_special_tokens=True)
    return "".join(ch for ch in text if ch.isalnum() or ch.isspace()).strip()

In [33]:
def cot_prompt(query):
    # Chain-of-Thought (Jagerman et al., 2023): elicit step-by-step reasoning,
    # whose terms serve as the query expansion.
    return f"Answer the following query:\n{query}\nGive the rationale before answering."

In [34]:
# sanity check on one query
demo = train_queries.iloc[1]["query"]
print("Query    :", demo)
print("CoT out  :", llm_generate(cot_prompt(demo)))

Query    : who sang what in the world's come over you
CoT out  : The worlds come over you is a song by American singersongwriter Taylor Swift Taylor Swift sang the song Come Over You in the worlds come over you The final answer Taylor Swift


#  Query expansion

In [35]:
def expand_topics_llm(topics, tag, prompt_fn, force=False, max_new_tokens=128, query_repeat=5):
    cache_path = cache_dir / f"llm_qe_{tag}.csv"
    time_path  = cache_dir / f"llm_qe_{tag}_time.json"
    if cache_path.exists() and not force:
        out = pd.read_csv(cache_path, dtype={"qid": str})
        out["query"]          = out["query"].fillna("").astype(str)
        out["expansion_text"] = out["expansion_text"].fillna("")
        print(f"Loaded LLM QE cache: {cache_path}")
        return out

    gen_texts, expanded = [], []
    t0 = time.time()
    for q in tqdm(topics["query"].tolist(), desc=f"LLM QE [{tag}]"):
        g = llm_generate(prompt_fn(q), max_new_tokens=max_new_tokens)
        gen_texts.append(g)
        expanded.append((" ".join([q] * query_repeat) + " " + g).strip())
    mean_ms = 1000.0 * (time.time() - t0) / max(len(topics), 1)

    out = topics.copy()
    out["orig_query"]     = out["query"]
    out["expansion_text"] = gen_texts
    out["query"]          = expanded
    out.to_csv(cache_path, index=False)
    with open(time_path, "w") as f:
        json.dump({"llm": LLM_NAME, "strategy": tag, "mean_prompt_ms": mean_ms,
                   "n": int(len(topics))}, f, indent=4)
    print(f"Saved {cache_path}  (mean {mean_ms:.1f} ms/query)")
    return out

In [37]:
llm_cot = expand_topics_llm(train_queries, f"cot_{LLM_TAG}", cot_prompt, max_new_tokens=128)
print("Original:", llm_cot.iloc[1]["orig_query"])
print("Expanded:", llm_cot.iloc[1]["query"])

Loaded LLM QE cache: /home/tlvj/msc_datalogi/2_semester/se/results/llm_qe_cot_flan_t5_base.csv
Original: who sang what in the world's come over you
Expanded: who sang what in the world's come over you who sang what in the world's come over you who sang what in the world's come over you who sang what in the world's come over you who sang what in the world's come over you The worlds come over you is a song by American singersongwriter Taylor Swift Taylor Swift sang the song Come Over You in the worlds come over you The final answer Taylor Swift


In [38]:
def make_qe_pipeline(expanded_df):
    qmap = dict(zip(expanded_df["qid"].astype(str), expanded_df["query"].astype(str)))
    rewrite = pt.apply.query(lambda r: qmap.get(str(r["qid"]), r["query"]))
    return rewrite >> pt.rewrite.tokenise() >> backbone_second

In [42]:
metrics = [nDCG@5, nDCG@10, nDCG@20, RR@10, P@5, P@10, P@20, R@5, R@10, R@20]
results_cache_path = cache_dir / f"llm_qe_cot_{LLM_TAG}_results.csv"

In [43]:
def run_cot_experiment(force=False):
    if results_cache_path.exists() and not force:
        out = pd.read_csv(results_cache_path)
        print(f"Loaded results cache: {results_cache_path}")
        return out
    out = pt.Experiment(
        [backbone_second, lm_baseline, make_qe_pipeline(llm_cot)],
        train_queries, train_qrels,
        eval_metrics=metrics,
        names=["BM25 baseline", "Hiemstra_LM baseline", "BM25 + LLM QE (CoT)"],
        filter_by_qrels=True, round=4,
    )
    out.to_csv(results_cache_path, index=False)
    print(f"Saved results to {results_cache_path}")
    return out

In [44]:
results = run_cot_experiment()

Saved results to /home/tlvj/msc_datalogi/2_semester/se/results/llm_qe_cot_flan_t5_base_results.csv


In [45]:
with open(cache_dir / f"llm_qe_cot_{LLM_TAG}_time.json") as f:
    t = json.load(f)
print(f"LLM prompting time: {t['mean_prompt_ms']:.1f} ms/query ({t['llm']})")
results

LLM prompting time: 525.5 ms/query (google/flan-t5-base)


,name,P@5,P@10,P@20,R@5,R@10,R@20,nDCG@5,nDCG@10,nDCG@20,RR@10
0,BM25 baseline,0.1727,0.1050,0.0607,0.4816,0.5894,0.6849,0.4125,0.4516,0.4778,0.4684
1,Hiemstra_LM baseline,0.1695,0.1048,0.0606,0.4701,0.5851,0.6815,0.3997,0.4411,0.4676,0.4601
2,BM25 + LLM QE (CoT),0.1719,0.1051,0.0608,0.4799,0.5895,0.6869,0.4116,0.4512,0.4780,0.4689
